In [0]:
# Quick connection test — no spark.conf.set needed now
SILVER_PATH = "abfss://silver@retailbankingdl.dfs.core.windows.net"

df = spark.read.format("delta").load(f"{SILVER_PATH}/transactions_cleaned")
print(f"✅ Connected — {df.count():,} rows in transactions_cleaned")

In [0]:
from pyspark.sql.functions import col

SILVER_PATH = "abfss://silver@retailbankingdl.dfs.core.windows.net"

# Load tables once
df_t = spark.read.format("delta").load(f"{SILVER_PATH}/transactions_cleaned")
df_c = spark.read.format("delta").load(f"{SILVER_PATH}/customers_scd2")

# Test runner
passed = 0
failed = 0
errors = []

def run_test(name, fn):
    global passed, failed
    try:
        fn()
        print(f"  ✅ PASSED — {name}")
        passed += 1
    except AssertionError as e:
        print(f"  ❌ FAILED — {name}: {e}")
        failed += 1
        errors.append(name)
    except Exception as e:
        print(f"  ❌ ERROR  — {name}: {str(e)[:80]}")
        failed += 1
        errors.append(name)

print("=" * 60)
print("Running Silver Layer Data Quality Tests")
print("=" * 60)

# Transactions tests
run_test("transactions_not_empty",            lambda: df_t.count() > 0)
run_test("transactions_min_row_count",        lambda: df_t.count() >= 280_000)
run_test("transactions_no_null_sk",           lambda: df_t.filter(col("transaction_sk").isNull()).count() == 0)
run_test("transactions_amounts_positive",     lambda: df_t.filter(col("transaction_amount") <= 0).count() == 0)
run_test("transactions_no_null_amounts",      lambda: df_t.filter(col("transaction_amount").isNull()).count() == 0)
run_test("transactions_fraud_label_valid",    lambda: df_t.filter(~col("is_fraud").isin(0,1)).count() == 0)
run_test("transactions_dq_passed_rate",       lambda: df_t.filter(col("dq_passed")==True).count()/df_t.count() >= 0.99)
run_test("transactions_fraud_rate_realistic", lambda: 0.001 <= df_t.filter(col("is_fraud")==1).count()/df_t.count() <= 0.01)
run_test("transactions_source_populated",     lambda: df_t.filter(col("silver_source").isNull()).count() == 0)
run_test("transactions_date_populated",       lambda: df_t.filter(col("silver_ingestion_date").isNull()).count() == 0)

# Customers SCD2 tests
run_test("customers_not_empty",              lambda: df_c.count() > 0)
run_test("customers_no_null_ids",            lambda: df_c.filter(col("customer_id").isNull()).count() == 0)
run_test("customers_sk_unique",              lambda: df_c.count() == df_c.select("customer_sk").distinct().count())
run_test("customers_one_current_per_id",     lambda: df_c.filter(col("is_current")==True).groupBy("customer_id").count().filter(col("count")>1).count() == 0)
run_test("customers_effective_dates_valid",  lambda: df_c.filter(col("effective_from") >= col("effective_to")).count() == 0)
run_test("customers_current_open_end_date",  lambda: df_c.filter((col("is_current")==True)&(col("effective_to")!="9999-12-31")).count() == 0)
run_test("customers_historical_closed",      lambda: df_c.filter((col("is_current")==False)&(col("effective_to")=="9999-12-31")).count() == 0)
run_test("customers_risk_rating_valid",      lambda: df_c.filter(~col("risk_rating").isin("LOW","MEDIUM","HIGH")).count() == 0)
run_test("customers_credit_limit_positive",  lambda: df_c.filter(col("credit_limit") <= 0).count() == 0)
run_test("customers_point_in_time_query",    lambda: df_c.filter((col("effective_from")<="2022-06-01")&(col("effective_to")>"2022-06-01")).groupBy("customer_id").count().filter(col("count")>1).count() == 0)

print("=" * 60)
print(f"Results: {passed} passed, {failed} failed")
if failed == 0:
    print("✅ ALL 20 TESTS PASSED — Silver layer is production-ready")
else:
    print(f"❌ FAILED: {errors}")
print("=" * 60)